In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import math

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Linear noise schedule as in original DDPM paper
def linear_beta_schedule(timesteps, beta_start=0.0001, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, timesteps, device=device)

# Parameters
T = 1000  # Total diffusion steps
beta = linear_beta_schedule(T)  # Linear schedule
alpha = 1.0 - beta
alpha_cumprod = torch.cumprod(alpha, dim=0)
alpha_cumprod_prev = F.pad(alpha_cumprod[:-1], (1, 0), value=1.0)

# Variance at each step
sqrt_recip_alphas = torch.sqrt(1.0 / alpha)
sqrt_alphas_cumprod = torch.sqrt(alpha_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alpha_cumprod)

# Posterior variance (q(x_{t-1} | x_t, x_0))
posterior_variance = beta * (1.0 - alpha_cumprod_prev) / (1.0 - alpha_cumprod)
posterior_mean_coef1 = beta * torch.sqrt(alpha_cumprod_prev) / (1.0 - alpha_cumprod)
posterior_mean_coef2 = (1.0 - alpha_cumprod_prev) * torch.sqrt(alpha) / (1.0 - alpha_cumprod)

# Forward diffusion process
def q_sample(x_0, t, noise=None):
    """
    Forward diffusion process: q(x_t | x_0)
    x_0: clean data
    t: timestep
    """
    if noise is None:
        noise = torch.randn_like(x_0)

    sqrt_alphas_cumprod_t = sqrt_alphas_cumprod[t][:, None, None, None]
    sqrt_one_minus_alphas_cumprod_t = sqrt_one_minus_alphas_cumprod[t][:, None, None, None]

    return sqrt_alphas_cumprod_t * x_0 + sqrt_one_minus_alphas_cumprod_t * noise, noise

# Simple sinusoidal position embeddings
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

# Simple U-Net for noise prediction
class SimpleUnet(nn.Module):
    def __init__(self):
        super().__init__()
        # Time embedding
        time_emb_dim = 256
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.GELU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim),
        )

        # Initial convolution
        self.conv_in = nn.Conv2d(1, 64, 3, padding=1)

        # Downsampling
        self.down1 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.down2 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Middle section
        self.middle = nn.Sequential(
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )

        # Time embedding projections
        self.time_emb_mid = nn.Sequential(
            nn.GELU(),
            nn.Linear(time_emb_dim, 256)
        )

        # Upsampling
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(512, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(256, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        # Final convolution
        self.conv_out = nn.Conv2d(128, 1, 3, padding=1)

    def forward(self, x, t):
        # Time embedding
        t_emb = self.time_mlp(t)

        # Initial convolution
        x1 = self.conv_in(x)

        # Downsample
        x2 = self.down1(x1)
        x3 = self.down2(x2)

        # Middle with time embedding
        x3_t = self.middle(x3)
        t_emb_mid = self.time_emb_mid(t_emb)[:, :, None, None].repeat(1, 1, x3.shape[2], x3.shape[3])
        x3_t = x3_t + t_emb_mid

        # Upsample with skip connections
        x = self.up1(torch.cat([x3_t, x3], dim=1))
        x = self.up2(torch.cat([x, x2], dim=1))

        # Output
        x = self.conv_out(torch.cat([x, x1], dim=1))

        return x

# Data preparation
def get_data_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])
    dataset = datasets.MNIST('.', download=True, train=True, transform=transform)
    return DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0)

# Training function
def train_ddpm(model, dataloader, optimizer, epochs=30):
    losses = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for step, (x, _) in enumerate(dataloader):
            x = x.to(device)
            batch_size = x.shape[0]

            # Sample random timesteps
            t = torch.randint(0, T, (batch_size,), device=device).long()

            # Get noisy images and noise
            noisy_x, noise = q_sample(x, t)

            # Predict noise
            noise_pred = model(noisy_x, t)

            # Calculate loss
            loss = F.mse_loss(noise_pred, noise)

            # Optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            if step % 100 == 0:
                print(f"Epoch {epoch+1}/{epochs}, Step {step}, Loss: {loss.item():.6f}")

        avg_loss = epoch_loss / len(dataloader)
        losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.6f}")

        # Save model periodically
        if (epoch + 1) % 5 == 0:
            torch.save(model.state_dict(), f"ddpm_model_epoch_{epoch+1}.pt")

    return losses

# Original DDPM sampling (not DDIM)
@torch.no_grad()
def p_sample(model, x, t):
    """
    Sample from p(x_{t-1} | x_t)
    """
    t_index = t[0].item()
    betas_t = beta[t_index]
    sqrt_one_minus_alphas_cumprod_t = sqrt_one_minus_alphas_cumprod[t_index]
    sqrt_recip_alphas_t = sqrt_recip_alphas[t_index]

    # Equation 11 in the paper
    # Use the model to predict the mean
    model_mean = sqrt_recip_alphas_t * (
        x - betas_t * model(x, t) / sqrt_one_minus_alphas_cumprod_t
    )

    if t_index == 0:
        return model_mean
    else:
        posterior_variance_t = posterior_variance[t_index]
        noise = torch.randn_like(x)
        return model_mean + torch.sqrt(posterior_variance_t) * noise

@torch.no_grad()
def p_sample_loop(model, shape):
    """
    Generate samples from the model and save intermediate steps
    """
    device = next(model.parameters()).device

    # Start from pure noise
    img = torch.randn(shape, device=device)
    intermediate_images = [img.cpu()]  # Save initial noise

    # Gradually denoise
    for t in reversed(range(T)):
        time_tensor = torch.full((shape[0],), t, device=device, dtype=torch.long)
        img = p_sample(model, img, time_tensor)

        # Save intermediate images at specific steps
        if t % 100 == 0 or t == 0:  # Save every 100 steps and the final step
            intermediate_images.append(img.cpu())

    return img, intermediate_images

# Generate sample images with intermediate steps
@torch.no_grad()
def generate_samples(model, n=16):
    model.eval()
    final_samples, intermediate_images = p_sample_loop(model, shape=(n, 1, 28, 28))

    # Create a grid of intermediate steps
    num_steps = len(intermediate_images)
    fig, axes = plt.subplots(num_steps, 1, figsize=(10, 5*num_steps))

    for i, imgs in enumerate(intermediate_images):
        # Create a grid of images for this step
        grid = torch.cat([imgs[j] for j in range(n)], dim=2)
        grid = grid.permute(1, 2, 0).cpu().numpy()

        # Plot the grid
        axes[i].imshow(grid.squeeze(), cmap='gray')
        axes[i].axis('off')
        axes[i].set_title(f'Step {T - i*100 if i < num_steps-1 else 0}')

    plt.tight_layout()
    plt.savefig('ddpm_samples_intermediate.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Also save the final samples
    grid = torch.cat([final_samples[i] for i in range(n)], dim=2)
    grid = grid.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=(10, 5))
    plt.imshow(grid.squeeze(), cmap='gray')
    plt.axis('off')
    plt.title('Final Samples')
    plt.tight_layout()
    plt.savefig('ddpm_samples_final.png', dpi=300)
    plt.show()

    return final_samples

if __name__ == '__main__':
    # Initialize model
    model = SimpleUnet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

    # Get data loader
    dataloader = get_data_loader()

    # Train model
    losses = train_ddpm(model, dataloader, optimizer, epochs=100)

    # Plot losses
    plt.figure(figsize=(10, 5))
    plt.plot(losses)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.savefig('ddpm_loss.png')

    # Generate samples
    samples = generate_samples(model, n=16)